# 01 — Análise Exploratória do Corpus

Implementa a Fase 18 do plano de elaboração (ver `PLANO-ELABORACAO.md`):
distribuição de classes, comprimento de texto, nuvens de palavras e
n-gramas mais frequentes do corpus rotulado de tweets em português.

Toda a lógica de carregamento/visualização vem de `src/` (`data.loader`,
`visualization.*`, `features.lexical`); este notebook apenas orquestra e
narra os resultados — nenhuma lógica de produção é definida aqui (ver
`CLAUDE.md`, seção "Notebooks").

**Pré-requisito**: as etapas `preprocessing` e `labeling` já devem ter
sido executadas (`uv run python src/main.py --stage preprocessing` e
`--stage labeling`), populando `paths.labeled_corpus_file`.

In [ ]:
import sys
from pathlib import Path

# `notebooks/` não fica dentro de `src/` (raiz de importação do projeto —
# ver CLAUDE.md, "Import style"), então o diretório precisa ser inserido
# em `sys.path` manualmente antes de qualquer import próprio.
sys.path.insert(0, str(Path.cwd().parent / "src"))

from config.paths import load_project_paths
from constants.labels import SENTIMENT_CLASSES
from data.loader import load_labeled_corpus
from features.lexical import build_document_frequencies, extract_ngrams
from preprocessing.tokenization import tokenize_and_normalize
from visualization.distributions import plot_class_distribution, plot_text_length_distribution
from visualization.ngrams import plot_top_ngrams_bar
from visualization.theme import apply_project_theme, save_figure
from visualization.wordcloud import generate_sentiment_wordcloud

apply_project_theme()
paths = load_project_paths()

## Carregamento do corpus rotulado

In [ ]:
labeled_corpus = load_labeled_corpus(paths.labeled_corpus_file)
print(f"Total de tweets rotulados: {labeled_corpus.height}")
labeled_corpus.head()

## Distribuição das classes de sentimento

Um forte desbalanceamento entre classes exige atenção na escolha de
métricas (F1-macro/MCC, ver `CLAUDE.md` -> "Project-Specific Overrides")
e no desenho da validação (estratificação).

In [ ]:
class_distribution_figure = plot_class_distribution(labeled_corpus["sentiment_label"].to_list())
save_figure(class_distribution_figure, "distribuicao_classes", directory=paths.reports_figures_dir)

## Distribuição do comprimento dos textos

Comprimentos muito díspares entre classes podem indicar um viés de coleta
(ex.: reclamações mais longas que elogios) que vale registrar no
datasheet do corpus (`reports/datasheets/`).

In [ ]:
text_lengths = [len(text) for text in labeled_corpus["text"].to_list()]
text_length_figure = plot_text_length_distribution(
    text_lengths, class_labels=labeled_corpus["sentiment_label"].to_list()
)
save_figure(
    text_length_figure, "distribuicao_comprimento_texto", directory=paths.reports_figures_dir
)

## Tokenização para as análises de vocabulário

Reaproveita o mesmo tokenizador usado no pipeline de features
(`preprocessing.tokenization.tokenize_and_normalize`), para que as
frequências observadas aqui reflitam o vocabulário que os modelos
clássicos realmente enxergam.

In [ ]:
tokens_by_row = [tokenize_and_normalize(text) for text in labeled_corpus["text"].to_list()]
labeled_corpus = labeled_corpus.with_columns(tokens=tokens_by_row)

## Nuvens de palavras por classe de sentimento

A frequência usada é a frequência documental (`build_document_frequencies`
— número de tweets em que cada palavra aparece), menos sensível a um único
tweet repetitivo do que a contagem bruta de ocorrências.

In [ ]:
for sentiment_class in SENTIMENT_CLASSES:
    class_tokens = labeled_corpus.filter(labeled_corpus["sentiment_label"] == sentiment_class)[
        "tokens"
    ].to_list()
    word_frequencies = build_document_frequencies(class_tokens)
    wordcloud_figure = generate_sentiment_wordcloud(
        word_frequencies, title=f"Nuvem de Palavras — {sentiment_class}"
    )
    save_figure(
        wordcloud_figure, f"nuvem_palavras_{sentiment_class}", directory=paths.reports_figures_dir
    )

## Bigramas mais frequentes por classe de sentimento

Bigramas costumam capturar padrões que uma única palavra não revela (ex.:
negações como "não gostei", já marcadas pelo tokenizador — ver
`preprocessing.tokenization.mark_negation_scope`).

In [ ]:
for sentiment_class in SENTIMENT_CLASSES:
    class_tokens = labeled_corpus.filter(labeled_corpus["sentiment_label"] == sentiment_class)[
        "tokens"
    ].to_list()
    class_bigrams = [extract_ngrams(tokens, ngram_range=(2, 2)) for tokens in class_tokens]
    bigram_frequencies = build_document_frequencies(class_bigrams)
    ngram_figure = plot_top_ngrams_bar(
        bigram_frequencies, top_n=15, title=f"Bigramas Mais Frequentes — {sentiment_class}"
    )
    save_figure(ngram_figure, f"bigramas_{sentiment_class}", directory=paths.reports_figures_dir)

## Conclusões

Registrar aqui, após a primeira execução: (1) o grau de desbalanceamento
entre classes e sua implicação para a estratégia de validação; (2)
diferenças relevantes de comprimento de texto entre classes; (3) termos e
bigramas que se destacam por classe e que justifiquem features léxicas
específicas (`src/features/lexical.py`) ou hipóteses a investigar via
HypotheSAEs (`notebooks/02_diagnostico_rotulagem.ipynb`).